# EMG-Robust demo

This notebook runs the real perturbation code from `emg_robust/perturbations.py` on a **synthetic** EMG signal and a tiny, untrained **toy model**, just to show the toolkit's mechanics end to end in under a minute.

It does **not** reproduce the paper's numbers — those come from the real emg2pose dataset and checkpoints. For those:

- `python scripts/make_report.py` rebuilds Table 1 and Fig. 2 from the shipped `results/*.csv` in a few seconds (no dataset, checkpoints or GPU needed — see the main README's "Quick start").
- `python scripts/run_sweep.py --model neuropose --user 1` runs the real 37-config sweep against an official checkpoint and held-out session (see the main README's "Run a sweep").

**This notebook's own dependencies:** NumPy, Matplotlib, pandas, and PyTorch (CPU is fine — no GPU, no dataset download, no checkpoints). `pip install torch numpy matplotlib pandas` is all it needs beyond the repo itself.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` to find the EMG-Robust repo root (the folder
    containing emg_robust/), so this works whether Jupyter's cwd is the
    repo root or the notebooks/ folder."""
    for p in [start, *start.parents]:
        if (p / "emg_robust").is_dir():
            return p
    raise FileNotFoundError("Could not find the 'emg_robust' package above the current directory")


sys.path.insert(0, str(find_repo_root(Path.cwd())))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from emg_robust import perturbations

torch.manual_seed(0)
np.random.seed(0)

## 1. A synthetic EMG window

16 channels, 5 seconds at 2 kHz (10,000 samples) — the same window shape used throughout EMG-Robust. Each channel is a handful of sinusoidal "bursts" of activity plus its own noise floor, so channels differ in both amplitude and σ<sub>c</sub> (the per-channel, per-window std that the noise perturbation scales by). This is only meant to *look* EMG-like; it carries no biomechanical meaning.

In [ ]:
def make_synthetic_emg(n_channels=16, length=10_000, fs=2_000, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(length) / fs
    emg = np.zeros((n_channels, length), dtype=np.float32)
    for c in range(n_channels):
        n_bursts = rng.integers(2, 5)
        for _ in range(n_bursts):
            centre = rng.uniform(0, t[-1])
            width = rng.uniform(0.15, 0.4)
            freq = rng.uniform(60, 150)  # Hz, roughly EMG-band
            envelope = np.exp(-0.5 * ((t - centre) / width) ** 2)
            emg[c] += envelope * np.sin(2 * np.pi * freq * t + rng.uniform(0, 2 * np.pi))
        emg[c] += rng.normal(0, 0.05 * (c + 1) / n_channels, size=length)  # channel-dependent noise floor
    return torch.from_numpy(emg).unsqueeze(0)  # [1, n_channels, length]


emg_clean = make_synthetic_emg()
print("emg_clean shape:", tuple(emg_clean.shape))

fig, ax = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
for i, ch in enumerate([0, 7, 15]):
    ax[i].plot(emg_clean[0, ch].numpy())
    ax[i].set_ylabel(f"ch {ch}")
ax[-1].set_xlabel("sample")
fig.suptitle("Synthetic EMG — 3 of 16 channels")
plt.tight_layout()
plt.show()

## 2. The four perturbations, at their severe level

Each call below is the *exact* function from `emg_robust/perturbations.py` — the same code `scripts/run_sweep.py` uses on real data.

In [ ]:
# Additive noise: x_c[t] + alpha * sigma_c * N(0,1)
rng = np.random.default_rng(seed=0)  # created once, as the real sweep does -- see add_gaussian_noise's docstring
emg_noisy = perturbations.add_gaussian_noise(emg_clean, level_frac=0.6, rng=rng)

ch = 0
sigma_c = emg_clean[0, ch].std().item()
plt.figure(figsize=(9, 2.5))
plt.plot(emg_clean[0, ch].numpy(), label="clean", alpha=0.8)
plt.plot(emg_noisy[0, ch].numpy(), label="+ noise (α=0.6)", alpha=0.7)
plt.title(f"Additive noise, severe (α=0.6) — channel {ch}, σ_c={sigma_c:.3f}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Channel dropout: pick k channels from (n_channels, seed) alone, then zero them
channels = perturbations.select_dropout_channels(n_channels=4, seed=0)
emg_dropped = perturbations.apply_channel_mask(emg_clean, channels)
print("channels zeroed:", channels)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].imshow(emg_clean[0].numpy(), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax[0].set_title("clean (all 16 channels)")
ax[1].imshow(emg_dropped[0].numpy(), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax[1].set_title(f"dropout, severe (k=4): channels {channels} zeroed")
for a in ax:
    a.set_xlabel("sample")
    a.set_ylabel("channel")
    a.set_yticks(range(16))
for c in channels:
    ax[1].get_yticklabels()[c].set_color("red")
    ax[1].get_yticklabels()[c].set_fontweight("bold")
plt.tight_layout()
plt.show()

In [ ]:
# Amplitude scaling: x_c[t] * scale
emg_scaled = perturbations.amplitude_scale(emg_clean, scale=0.25)

ch = 0
plt.figure(figsize=(9, 2.5))
plt.plot(emg_clean[0, ch].numpy(), label="clean", alpha=0.8)
plt.plot(emg_scaled[0, ch].numpy(), label="× 0.25", alpha=0.8)
plt.title(f"Amplitude scaling, severe (×0.25) — channel {ch}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Temporal shift: delay by `shift_samples`, zero-padded at the start (200 samples = 100 ms @ 2 kHz)
emg_shifted = perturbations.temporal_shift(emg_clean, shift_samples=200)

ch = 0
plt.figure(figsize=(9, 2.5))
plt.plot(emg_clean[0, ch, :1000].numpy(), label="clean", alpha=0.8)
plt.plot(emg_shifted[0, ch, :1000].numpy(), label="delayed 200 samples (100 ms)", alpha=0.8)
plt.title(f"Temporal shift, severe (100 ms) — channel {ch}, first 1000 samples")
plt.legend()
plt.tight_layout()
plt.show()

## 3. The same config factory the real sweep uses

`make_perturbation(perturbation, value, seed)` is exactly what `scripts/run_sweep.py` calls once per sweep row. It is the only thing a config needs to turn into a `perturb_fn` — this reproduces the dropout call above without repeating the `(n_channels, seed)` logic by hand.

In [ ]:
perturb_fn, dropped_str = perturbations.make_perturbation("dropout", value=4, seed=0)
emg_via_factory = perturb_fn(emg_clean)
print("dropped_channels string (as stored in results/*.csv):", dropped_str)
assert torch.equal(emg_via_factory, emg_dropped)
print("Matches the direct call above ✓")

## 4. A minimal end-to-end sweep, with a toy model

To show the full loop — perturb, run the model, record a row — without the real emg2pose dataset, checkpoints or compute, this section plugs in a tiny, untrained, fixed-random linear layer as a stand-in "pose decoder" and measures how far its output drifts from its own clean-input output.

This drift number is **not** the paper's landmark error, and this toy decoder has no connection to NeuroPose, vemg2pose, or any real pose model — it is only here to give the sweep loop something to measure. But the loop structure (one row per perturbation × level × seed, same column names as `results/*.csv`) is the same one `emg_robust/sweep.py` runs for real, at full scale (37 configs, 5 seeds, real checkpoints), via `scripts/run_sweep.py`.

In [ ]:
toy_decoder = torch.nn.Linear(16, 20)
toy_decoder.eval()


def toy_pose(emg):
    """[1, 16, T] -> [1, T, 20]: a stand-in 'pose' decoder, never trained,
    just dense enough to mix channels so a perturbation has a visible effect
    on its output."""
    with torch.no_grad():
        return toy_decoder(emg.permute(0, 2, 1))


clean_output = toy_pose(emg_clean)

DEMO_CONFIGS = [
    ("noise", "mild", 0.1), ("noise", "moderate", 0.3), ("noise", "severe", 0.6),
    ("dropout", "mild", 1), ("dropout", "moderate", 2), ("dropout", "severe", 4),
    ("amplitude", "mild", 0.75), ("amplitude", "moderate", 0.5), ("amplitude", "severe", 0.25),
    ("temporal_shift", "mild", 40), ("temporal_shift", "moderate", 100), ("temporal_shift", "severe", 200),
]  # one seed each, for speed -- scripts/run_sweep.py runs 5 seeds for noise/dropout

rows = []
for perturbation, level, value in DEMO_CONFIGS:
    perturb_fn, dropped = perturbations.make_perturbation(perturbation, value, seed=0)
    perturbed_output = toy_pose(perturb_fn(emg_clean))
    drift = (perturbed_output - clean_output).pow(2).mean().sqrt().item()
    rows.append(dict(perturbation=perturbation, level=level, value=value,
                      pose_drift_rmse=drift, dropped_channels=dropped))

demo_df = pd.DataFrame(rows)
demo_df

In [ ]:
pivot = demo_df.pivot(index="perturbation", columns="level", values="pose_drift_rmse")
pivot = pivot[["mild", "moderate", "severe"]]

fig, ax = plt.subplots(figsize=(5, 3))
im = ax.imshow(pivot.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(3))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center", color="white")
ax.set_title("Toy pose-drift RMSE (synthetic model + data — illustrative only)")
plt.colorbar(im, label="RMSE (arbitrary units)")
plt.tight_layout()
plt.show()

**Reading this toy heatmap:** for noise, dropout and amplitude, drift increases from mild to severe, as you'd expect. Temporal shift usually won't — this toy decoder has no memory across time (it maps each timestep's EMG to an output independently), so delaying the signal just compares the decoder's output at two different points in the same bursty synthetic trace, which doesn't have to get "worse" as the delay grows. A real pose model has temporal context and is evaluated on real motion, which is why the paper's own results (`results/`, Table 1) show a different — and more consistent — pattern across perturbations than this toy demo does. Don't read anything into the actual numbers here beyond "the mechanism runs."

## Next steps

- **Real numbers:** `python scripts/make_report.py` reproduces Table 1 and Fig. 2 from the shipped `results/*.csv` in a few seconds (no dataset or GPU needed either).
- **Real sweep:** `python scripts/run_sweep.py --model neuropose --user 1` runs the full 37-config sweep against the official emg2pose checkpoint and held-out session (see the main README's "Run a sweep").
- **Your own model:** swap `toy_decoder`/`toy_pose` above for your model wrapped in the `ModelAdapter` interface (see the main README's "Add your own model"), and use `emg_robust.sweep.build_configs()` / `run_sweep()` in place of `DEMO_CONFIGS` above to run the real 37-config, 5-seed sweep on it.